# Tutorial 4: Evaluating LLM Agents on Mathematical Reasoning

Welcome to the fourth tutorial in our AI Safety Evaluations course.

So far you have evaluated models as **passive responders** — the model reads a prompt,
produces an answer, and you score it. But many real-world AI systems are **agents**: they
observe, reason, act (e.g. call a tool), observe the result, and repeat. Evaluating agents
is harder because the model's behaviour is no longer a single forward pass — it is a
*multi-step trajectory* where mistakes compound and new failure modes (infinite loops,
tool misuse, hallucinated tool calls) appear.

In this tutorial you will build and evaluate a simple **ReAct agent** that solves
math problems by calling calculator and algebra tools. You will see first-hand how
scaffolding choices — prompts, tool sets, message limits, output formatting — affect
agent performance, and you will practice a basic dev/test workflow for iterating on
an agent without overfitting to your evaluation set. Think of it as a toy,
simplified version of a real elicitation pipeline like the
[METR Elicitation Protocol](https://evaluations.metr.org/elicitation-protocol/).

**What you'll learn:**

- Define custom tools for inspect_ai agents
- Build a ReAct agent and iteratively improve it on a dev set
- Develop intuition for how scaffolding choices affect agent performance

**By the end:** **You'll have a working agent evaluation pipeline and hands-on experience with the kind of iteration loop used in real-world agent evals.**

## 1. Setup

**Model choice.** We recommend picking a model that isn't too powerful — ideally one that occasionally
stumbles on arithmetic so you can observe the effect of giving it tools. The examples
below use `qwen2.5:3b`, but feel free to swap in any model you have access to (just make sure it supports tool calling). The
main goal is to see how well the model uses the tools it's given, so don't worry if
the tool-augmented score ends up lower than plain generation — that's a valid and
interesting finding, not a sign that something is broken. Conversely, if your model
solves everything perfectly even without tools, consider switching to a harder dataset
(e.g. the full MATH instead of MATH-500, or a competition-math set like AIME) — just
make sure to note this in your write-up.

> **Resource note:** Agent evaluations generate many more LLM calls than simple Q&A evals
> because each problem may involve multiple reasoning steps. All `eval()` calls in this
> notebook use a `limit` parameter to cap the number of samples processed. Adjust
> `EVAL_LIMIT` and `MAX_MESSAGES` if your machine is slow.

In [1]:
!pip install sympy datasets scipy -q
print("✅ Installed: sympy, datasets, scipy")


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✅ Installed: sympy, datasets, scipy


In [5]:
import re
import os
import math
import random
from textwrap import dedent
from collections import defaultdict
from scipy.stats import norm

from inspect_ai import Task, eval
from inspect_ai.dataset import Sample, hf_dataset
from inspect_ai.agent import react
from inspect_ai.solver import generate, use_tools, system_message, TaskState
from inspect_ai.scorer import (
    Score, Target, model_graded_qa, scorer, accuracy, stderr, match, mean
)
from inspect_ai.tool import tool

%load_ext dotenv
%dotenv /root/.env

In [10]:
os.environ["OPENAI_API_KEY"] = os.environ["LITELLM_API_KEY"]
os.environ["OPENAI_BASE_URL"] = "https://srs-litellm.kontur.host/v1"

MODEL = 'openai/code-pro'

In [7]:
RANDOM_SEED = 42
EVAL_LIMIT = 30        # max samples per eval run (raise if your machine is fast)
MAX_MESSAGES = 20      # max back-and-forth messages per agent trajectory

A few helper functions for extracting and displaying results. You don't need to modify
these — just run the cell and move on.

In [8]:
def get_acc(log):
    """Extract accuracy (or mean) from the first scorer in a log."""
    m = log.results.scores[0].metrics
    return (m.get("accuracy") or m.get("mean")).value


def _first_score(sample):
    """Get the first Score object from a sample, regardless of scorer name."""
    scores = sample.scores
    first_key = list(scores.keys())[0]
    val = scores[first_key]
    return val[0] if isinstance(val, list) else val


def print_results(label, log):
    """Pretty-print per-sample results from an eval log."""
    acc = get_acc(log)
    print(f"{'=' * 60}")
    print(f"  {label}   accuracy: {acc:.0%}")
    print(f"{'=' * 60}")
    for i, sample in enumerate(log.samples, 1):
        sc = _first_score(sample)
        msgs = len(sample.messages)
        expl = (sc.explanation or "")[:60]
        print(f"  [{sc.value}] #{i:2d}  msgs={msgs:2d}  "
              f"target={sample.target[:20]:>20s}  {expl}")
    print()

## 2. Tools and Agent Architecture

### Why agents?

A standard LLM evaluation looks like this: you hand the model a question, it produces
an answer, and a scorer checks whether the answer is correct. The model has *one shot*.

But many practical AI systems need to **interact with the world** — search the web, run
code, query a database, or call an API — before they can answer. These systems are
called **agents**. An agent follows a loop:

1. **Observe** the current state (the question, plus any tool outputs so far).
2. **Think** about what to do next.
3. **Act** by calling a tool (or submitting a final answer).
4. **Observe** the tool's result, then go back to step 2.

This loop continues until the agent decides it has enough information to submit a final
answer, or until a safety limit (maximum number of steps) is reached.

### Why not just give the model tools and let it figure things out?

Because *access* to tools is not enough. The model also needs:

- A **system prompt** that tells it what tools exist and how to use them
- A **loop structure** that feeds tool results back so the model can reason about them
- A **stopping criterion** so the model knows when and how to submit its final answer
- **Message limits** to prevent runaway loops that burn tokens without progress

This combination of prompt + loop + stopping logic is called **scaffolding**, and it
can make or break agent performance. Whether an agent actually helps depends on the
task, the tools, the prompt, and the model — scaffolding is not a guaranteed win, but
understanding it is essential for evaluating agent systems.

### Approaches to giving a model tools

The simplest way to add tools in inspect_ai is the **`use_tools()` + `generate()`** pattern.
`use_tools()` registers a list of tool functions so the model can call them, and
`generate()` runs a **single generation**. The model may call tools during that generation,
but the scaffolding never interrupts — tool calls and the final answer are produced in
one continuous flow, with no structured pause to reconsider.
```python
solver = [
    system_message("You have access to calculator tools."),
    use_tools([add(), multiply()]),
    generate(),
]
```

This is easy to set up but fragile: if a tool returns something unexpected mid-generation,
the model is already committed to a line of reasoning and may not change course.

### The ReAct pattern

**ReAct** (Reason + Act) is a scaffolding pattern that introduces an explicit pause after
every tool call. The model reasons and calls one tool; the scaffolding then appends the
result to the context and starts a **fresh generation** — so the model reconsiders its plan
from scratch before deciding the next step. This closed feedback loop makes it much easier
to recover from unexpected tool results or multi-step reasoning chains.
```
┌─────────────────────────────────────────────────────────┐
│                    ReAct Agent Loop                     │
│                                                         │
│   ┌──────────┐    ┌──────────┐    ┌──────────┐          │
│   │  THINK   │───>│   ACT    │───>│ OBSERVE  │──┐       │
│   │          │    │          │    │          │  │       │
│   │ "I need  │    │ call     │    │ tool     │  │       │
│   │  to add  │    │ add(a,b) │    │ returns  │  │       │
│   │  these"  │    │          │    │ "111915" │  │       │
│   └──────────┘    └──────────┘    └──────────┘  │       │
│        ^                                        │       │
│        └────────────────────────────────────────┘       │
│                                                         │
│   Loop continues until the agent calls submit()         │
│   or the message limit is reached.                      │
└─────────────────────────────────────────────────────────┘
```

In **inspect_ai**, the `react()` solver implements this pattern. It:
1. Sends the problem with a system prompt describing available tools
2. Lets the model think and call one tool at a time
3. Appends the tool result to the context and starts a fresh generation
4. Repeats until the model calls `submit()` (a built-in action added automatically)

The `submit()` tool is special — it signals the end of the loop and its argument
becomes the agent's final answer. You never define `submit()` yourself; `react()`
adds it for you.

### Defining tools in inspect_ai

An agent needs *actions* it can take in the world. In inspect_ai, actions are
**tools** — Python functions the model can call during its reasoning loop.

The `@tool` decorator registers a function so that inspect_ai can:
1. **Describe** it to the model (via the docstring and type hints — these become the
   tool schema the model sees).
2. **Execute** it when the model emits a tool-call message and return the result.

The pattern is a factory function (decorated with `@tool`) that returns an `async`
inner function. The inner function's docstring and parameter annotations are what the
model actually sees, so clear names and descriptions directly affect agent performance.

Below we define four arithmetic tools ourselves. Notice how each one:
- Takes typed parameters with descriptive `Args:` docstrings
- Returns a **string** (tool outputs are always strings in inspect_ai)
- Wraps execution in `try/except` so the agent gets a readable error instead of a crash

> **Built-in tools:** inspect_ai also ships with ready-made tools for common agent tasks — you don't need to write these yourself:
> - `bash()` — run shell commands
> - `python()` — execute Python code
> - `web_search()` — search the web
> - `web_browser()` — full browser interaction
> - `text_editor()` — read and edit files
>
> For the full list see the [Tools section of the inspect_ai documentation](https://inspect.ai-safety-institute.org.uk/tools.html).
> In this tutorial we write our own tools from scratch to understand the mechanics,
> but in practice you will often mix custom tools with these built-in ones.

In [11]:
@tool
def add():
    async def execute(a: float, b: float) -> str:
        """Add two numbers.

        Args:
            a: First number.
            b: Second number.
        """
        try:
            return str(float(a) + float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def subtract():
    async def execute(a: float, b: float) -> str:
        """Subtract b from a.

        Args:
            a: Number to subtract from.
            b: Number to subtract.
        """
        try:
            return str(float(a) - float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def multiply():
    async def execute(a: float, b: float) -> str:
        """Multiply two numbers.

        Args:
            a: First number.
            b: Second number.
        """
        try:
            return str(float(a) * float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def divide():
    async def execute(a: float, b: float) -> str:
        """Divide a by b.

        Args:
            a: Dividend.
            b: Divisor (must not be zero).
        """
        try:
            b_val = float(b)
            if b_val == 0:
                return "Error: division by zero."
            return str(float(a) / b_val)
        except Exception as e:
            return f"Error: {e}"
    return execute

## Assignment 1: Create a `modular_arithmetic` tool

Cryptography and number theory problems often require modular arithmetic —
for example, computing $7^{1000} \mod 13$ as part of a larger proof.

Following the pattern above, implement a `modular_arithmetic` tool with a clear docstring.

In [12]:
@tool
def modular_arithmetic():
    async def execute(a: int, b: int) -> str:
        """Compute a modulo b (the remainder of a divided by b).
        Args:
            a: The dividend (integer).
            b: The modulus (positive integer, must not be zero).

        Returns:
            The remainder a mod b as a string. For negative a, the result
            is the non-negative remainder (Python's % semantics).
        """
        try:
            b_val = int(b)
            if b_val == 0:
                return "Error: modulus b cannot be zero."
            return str(int(a) % b_val)
        except Exception as e:
            return f"Error: {e}"
    return execute

Now let's see the tool in action. We run a small eval so the model attempts to use your tool. Don't worry if the model gets the answer wrong — what matters here is that the tool itself is correctly defined.

In [13]:
_test_samples_mod = [
    Sample(input="What is 1000000 mod 397? Reply with just the number.", target="354"),
    Sample(input="What is 100 mod 10? Reply with just the number.", target="0"),
]

_log_mod_test = eval(
    Task(
        dataset=_test_samples_mod,
        solver=react(
            prompt="You have a modular_arithmetic(a, b) tool. Use it, then submit the result.",
            tools=[modular_arithmetic()],
            attempts=1,
        ),
        scorer=match(numeric=True),
        message_limit=10,
    ),
    model=MODEL,
)[0]

print_results("modular_arithmetic tool test", _log_mod_test)

Output()

  modular_arithmetic tool test   accuracy: 100%
  [C] # 1  msgs= 5  target=                 354  354


354
  [C] # 2  msgs= 5  target=                   0  0


0



> **Checking tool usage.** Run **`inspect view`** in the terminal to see the full
> message trace for each sample — tool calls and their responses appear as explicit
> steps. Alternatively, inspect `log.samples[i].messages` directly: each tool
> call appears as a message with `role="assistant"` and a non-empty **`tool_calls`** field.

In [28]:
# компактно посмотреть по всем примерам

def inspect_all_samples(log):
    for i, sample in enumerate(log.samples):
        print(f"\nПример #{i}")
        print(f"Вопрос: {sample.input}")
        calls_in_sample = []
        for msg in sample.messages:
            if msg.role == "assistant" and msg.tool_calls:
                for tc in msg.tool_calls:
                    calls_in_sample.append(f"{tc.function}({tc.arguments})")
        if calls_in_sample:
            print(f"Вызовы инструментов: {', '.join(calls_in_sample)}")
        else:
            print("Инструменты не вызывались")

inspect_all_samples(_log_mod_test)


Пример #0
Вопрос: What is 1000000 mod 397? Reply with just the number.
Вызовы инструментов: modular_arithmetic({'a': 1000000, 'b': 397})

Пример #1
Вопрос: What is 100 mod 10? Reply with just the number.
Вызовы инструментов: modular_arithmetic({'a': 100, 'b': 10})


In [35]:
# более детально

def traces(log):
    for i, sample in enumerate(log.samples):
        print(f"\nПример #{i}")
        print(f"Вопрос: ```{sample.input}```")
        for msg in sample.messages:
            if msg.role == "system":
                continue  # пропускаем длинный system prompt
            elif msg.role == "user":
                print(f"[USER] ```{msg.content}```")
            elif msg.role == "assistant":
                if msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"[ASSISTANT] calls {tc.function}({tc.arguments})")
                    if msg.content:
                        print(f"Reasoning: ```{msg.content}```")
                else:
                    print(f"[ASSISTANT] answers: ```{msg.content}```")
            elif msg.role == "tool":
                print(f"[TOOL] returns: ```{msg.content}```")

traces(_log_mod_test)


Пример #0
Вопрос: ```What is 1000000 mod 397? Reply with just the number.```
[USER] ```What is 1000000 mod 397? Reply with just the number.```
[ASSISTANT] calls modular_arithmetic({'a': 1000000, 'b': 397})
Reasoning: ```I need to calculate 1000000 mod 397 to find the remainder when 1000000 is divided by 397.

Let me use the modular_arithmetic tool for this calculation.
```
[TOOL] returns: ```354```
[ASSISTANT] answers: ```354


354```

Пример #1
Вопрос: ```What is 100 mod 10? Reply with just the number.```
[USER] ```What is 100 mod 10? Reply with just the number.```
[ASSISTANT] calls modular_arithmetic({'a': 100, 'b': 10})
[TOOL] returns: ```0```
[ASSISTANT] answers: ```0


0```


In [21]:
# лог

for i, msg in enumerate(_log_mod_test.samples[0].messages):
    print(f"{i:2d}: {msg.role} -> {msg.model_dump_json(indent=2)}")

 0: system -> {
  "id": "KnvgK36pbeaapkqGCucKFZ",
  "content": "You have a modular_arithmetic(a, b) tool. Use it, then submit the result.\n\n\nYou are a helpful assistant attempting to submit the best possible answer. You have several tools available to help with finding the answer. You will see the result of tool calls right after sending the message. If you need to perform multiple actions, you can always send more messages with additional tool calls. Do some reasoning before your actions, describing what tool calls you are going to use and how they fit into your plan.\n\n\nWhen you have completed the task and have an answer, call the submit() tool to report it.\n",
  "source": null,
  "metadata": null,
  "role": "system"
}
 1: user -> {
  "id": "7RHw5VoJ8xRSbicExEcFAr",
  "content": "What is 1000000 mod 397? Reply with just the number.",
  "source": "input",
  "metadata": null,
  "role": "user",
  "tool_call_id": null
}
 2: assistant -> {
  "id": "TtvQDMSUPpGuEpZcuVU8uQ",
  "content

---
1. Did the model actually *use* your tool, or did it answer without using it?
   Open the eval log in the inspect_ai viewer (`inspect view`) and check the trace —
   you should see explicit tool call steps between the initial question and the final answer.
2. If the model skipped the tool, adjust the prompt in the `react()` call above and
   re-run until the model uses it. What did you change?

**Your answer:**

1. Да, использовала. В ячейках выше вывел, как именно
2. Ничего не менял, инструменты использовались

In [37]:
ARITH_TOOLS = [add(), subtract(), multiply(), divide(), modular_arithmetic()]

## 3. Toy Evaluation — Three Solver Architectures

Before touching a real benchmark, let's try out three different solver architectures
on a small set of hand-crafted problems. These 12 word problems use numbers large
enough that a model without tools might make arithmetic errors.

We will compare three solver architectures:

- **Plain generation** - reads the question, produces an answer in one shot;
  solver: `generate()`
- **Naive tool loop** - gets access to tools; `generate()` runs once and may call
  some tools; solver: `use_tools()` + `generate()`
- **ReAct agent** - explicit think-act-observe loop with a `submit()` action to stop;
  solver: `react()`
  
The goal here is to see how each architecture behaves — both in terms of accuracy
and how the solver actually runs — before moving to a real benchmark.

In [79]:
TOY_SAMPLES = [
    Sample(
        input=(
            "Найдите наименьшее натуральное число x, которое сравнимо с 39 по модулю 51."
        ),
        target="15",
    ),
    Sample(
        input=(
            "Найдите остаток от деления 735^{286} на 2431."
        ),
        target="2396",
    ),
    Sample(
        input=(
            "Найди наибольший общий делить чисел 641060580 и 702705960"
        ),
        target="60",
    ),
    Sample(
        input=(
            "A semiconductor factory produced 48,397 chips on Monday "
            "and 63,518 chips on Tuesday. How many chips were produced in total?"
        ),
        target="111915",
    ),
    Sample(
        input=(
            "A government reserve had 874,203 barrels of oil. "
            "After an emergency release, 295,867 barrels were distributed. "
            "How many barrels remain in the reserve?"
        ),
        target="578336",
    ),
    Sample(
        input=(
            "A logistics company ships 4,738 containers, each holding 2,659 units. "
            "How many units are shipped in total?"
        ),
        target="12598342",
    ),
    Sample(
        input=(
            "A national census counted 8,743,291 residents across 6,473 districts. "
            "If residents are distributed equally, how many full residents "
            "are assigned per district?"
        ),
        target="1350",
    ),
    Sample(
        input=(
            "A satellite completes a full orbit every 397 minutes. "
            "After exactly 1,000,000 minutes of operation, how many minutes "
            "have passed since the last complete orbit?"
        ),
        target="354",
    ),
    Sample(
        input=(
            "A hospital ordered 12,475 boxes of supplies at 387 dollars per box. "
            "They received a bulk discount of 843,750 dollars off the total. "
            "How much did the hospital pay after the discount?"
        ),
        target="3984075",
    ),
    Sample(
        input=(
            "A city has 14 times as many residents as municipal employees. "
            "If the total number of residents and employees together is 489,375, "
            "how many municipal employees does the city have?"
        ),
        target="32625",
    ),
    Sample(
        input=(
            "An airline flew 3,847 domestic flights and 2,964 international flights "
            "last month. Each flight used an average of 8,753 liters of fuel. "
            "How many liters of fuel were used in total?"
        ),
        target="59616683",
    ),
    Sample(
        input=(
            "A clock tower rings a bell every 1,873 seconds. "
            "After 10,000,000 seconds have elapsed since midnight, "
            "how many seconds ago did the bell last ring?"
        ),
        target="53",
    ),
    Sample(
        input=(
            "A farm harvested 247,839 kg of wheat and 184,672 kg of barley. "
            "The grain is loaded into trucks that carry exactly 4,750 kg each. "
            "How many full truckloads can be made from all the grain?"
        ),
        target="91",
    ),
    Sample(
        input=(
            "A global streaming platform has 1,847,293,847,291 seconds of video content. "
            "Given that a day has 86,400 seconds, how many full days of content "
            "does the platform have?"
        ),
        target="21380715",
    ),
    Sample(
        input=(
            "A country's economy grew by 3,847 dollars per citizen in a year. "
            "The country has 847,293,847 citizens. "
            "What was the total economic growth in dollars?"
        ),
        target="3259539429409",
    ),
]

## 3a. Approach 0: Plain generation (no tools)

The simplest baseline: give the model a system prompt and ask it to solve the problem
directly. The solver is just `generate()` — a single forward pass with no tool access.

We score with `match(numeric=True)`, which extracts the first number from the model's
response and compares it to the target.

In [80]:
SIMPLE_PROMPT = dedent("""    You are a math solver. Read the problem carefully, compute the answer,
    and respond with the final numeric result.
""")

log_toy_gen = eval(
    Task(
        dataset=TOY_SAMPLES,
        solver=[system_message(SIMPLE_PROMPT), generate()],
        scorer=match(numeric=True),
    ),
    model=MODEL,
)[0]

print_results("Approach 0: generate() only", log_toy_gen)

Output()

  Approach 0: generate() only   accuracy: 47%
  [I] # 1  msgs= 3  target=                  15  Нам нужно найти наименьшее натуральное число $ x $, такое чт
  [C] # 2  msgs= 3  target=                2396  Нам нужно найти остаток от деления $ 735^{286} $ на $ 2431 $
  [I] # 3  msgs= 3  target=                  60  Чтобы найти наибольший общий делитель (НОД) двух чисел 64106
  [C] # 4  msgs= 3  target=              111915  I need to find the total number of chips produced over the t
  [C] # 5  msgs= 3  target=              578336  I need to find how many barrels remain in the reserve after 
  [I] # 6  msgs= 3  target=            12598342  I need to find the total number of units shipped by multiply
  [C] # 7  msgs= 3  target=                1350  I need to find how many full residents are assigned per dist
  [C] # 8  msgs= 3  target=                 354  I need to find how many minutes have passed since the last c
  [I] # 9  msgs= 3  target=             3984075  I need to calculate the t

In [81]:
for sample in log_toy_gen.samples:
    score = _first_score(sample)
    print(f"Target: {sample.target}, Answer: {score.answer}, Value: {score.value}")

Target: 15, Answer: 39, Value: I
Target: 2396, Answer: 2396, Value: C
Target: 60, Answer: 0, Value: I
Target: 111915, Answer: 1.1192e+05, Value: C
Target: 578336, Answer: 5.7834e+05, Value: C
Target: 12598342, Answer: 1.2608e+07, Value: I
Target: 1350, Answer: 1350, Value: C
Target: 354, Answer: 354, Value: C
Target: 3984075, Answer: 3.9824e+06, Value: I
Target: 32625, Answer: 32625, Value: C
Target: 59616683, Answer: 6.0617e+07, Value: I
Target: 53, Answer: 253, Value: C
Target: 91, Answer: 90, Value: I
Target: 21380715, Answer: 2.1404e+07, Value: I
Target: 3259539429409, Answer: 3.2561e+12, Value: I


---
1. Did the model get everything right, or did it make arithmetic errors on the larger numbers?
2. If there were errors, what do you think caused them?

**Your answer:**

1. Нет, решено менее половины задач.
2. Проблемы в основном с большими числами. Для них использовались приближённые вычисления.

## 3b. Approach A: Naive tool loop

Now we give the model access to our arithmetic tools via `use_tools()`, followed by a
single `generate()`. The model *can* call tools, but the solver doesn't enforce any
structure: it may call one tool, multiple tools, or none at all, and it generates a
final answer in the same pass.

> In practice this pattern is rarely used on its own — without scaffolding, whether the
> model actually uses the tools is largely unpredictable.

In [82]:
NAIVE_LOOP_PROMPT = dedent("""    You are a math solver with access to calculator tools.
    Break each problem into arithmetic steps and call one tool per step.
""")

log_toy_naive = eval(
    Task(
        dataset=TOY_SAMPLES,
        solver=[
            system_message(NAIVE_LOOP_PROMPT),
            use_tools(ARITH_TOOLS),
            generate(),
        ],
        scorer=match(numeric=True),
    ),
    model=MODEL,
)[0]

print_results("Approach A: use_tools + generate (naive loop)", log_toy_naive)

Output()

  Approach A: use_tools + generate (naive loop)   accuracy: 87%
  [I] # 1  msgs= 5  target=                  15  Наименьшее натуральное число $ x $, которое сравнимо с 39 по
  [I] # 2  msgs=47  target=                2396  The remainder when $735^{286}$ is divided by 2431 is $\boxed
  [C] # 3  msgs=31  target=                  60  The result of $240 \mod 60$ is 0. Since we've reached a rema
  [C] # 4  msgs= 5  target=              111915  The total number of chips produced on Monday and Tuesday is 
  [C] # 5  msgs= 5  target=              578336  After the emergency release, there are 578,336 barrels remai
  [C] # 6  msgs= 5  target=            12598342  The total number of units shipped is 12,598,342.
  [C] # 7  msgs= 5  target=                1350  The division result shows that there are approximately 1350.
  [C] # 8  msgs= 5  target=                 354  After exactly 1,000,000 minutes of operation, 354 minutes ha
  [C] # 9  msgs= 7  target=             3984075  The hospital paid $

In [83]:
traces(log_toy_naive)


Пример #0
Вопрос: ```Найдите наименьшее натуральное число x, которое сравнимо с 39 по модулю 51.```
[USER] ```Найдите наименьшее натуральное число x, которое сравнимо с 39 по модулю 51.```
[ASSISTANT] calls modular_arithmetic({'a': 39, 'b': 51})
Reasoning: ```Чтобы найти наименьшее натуральное число $ x $, которое сравнимо с 39 по модулю 51, нужно решить сравнение:

$$
x \equiv 39 \pmod{51}
$$

Это означает, что $ x $ при делении на 51 даёт остаток 39. Наименьшее натуральное число, удовлетворяющее этому условию, — это просто 39, так как оно меньше 51 и даёт остаток 39 при делении на 51.

Таким образом, $ x = 39 $. 

Для проверки можно вычислить $ 39 \mod 51 $, чтобы убедиться, что остаток равен 39.
```
[TOOL] returns: ```39```
[ASSISTANT] answers: ```Наименьшее натуральное число $ x $, которое сравнимо с 39 по модулю 51, равно 39.```

Пример #1
Вопрос: ```Найдите остаток от деления 735^{286} на 2431.```
[USER] ```Найдите остаток от деления 735^{286} на 2431.```
[ASSISTANT] calls multi

---
1. Did having access to tools improve results compared to the baseline?
2. Did the model actually use the tools, or did it ignore them?

**Your answer:**
1. Да, результат стал гораздо лучше --- я был вынужден добавить несколько задач.
2. Да, я посмотрел лог и вижу, что инструменты использовались.

## 3c. Approach B: ReAct agent

Now let's use the `react()` solver — the full ReAct loop described in the introduction.
Notice the prompt explicitly tells the model to use tools and call `submit()` at the end.

In [84]:
REACT_PROMPT_V1 = dedent("""    You are a math solver with access to calculator tools.
    Break each problem into arithmetic steps and call one tool per step.
    Don't calculate anything without tools.
    After getting the final numeric result, call submit() with ONLY the number.
""")

log_toy_react = eval(
    Task(
        dataset=TOY_SAMPLES,
        solver=react(prompt=REACT_PROMPT_V1, tools=ARITH_TOOLS, attempts=1),
        scorer=match(numeric=True),
        message_limit=20,
    ),
    model=MODEL,
)[0]

print_results("Approach B: react() with simple prompt", log_toy_react)

Output()

  Approach B: react() with simple prompt   accuracy: 87%
  [I] # 1  msgs= 7  target=                  15  I've determined that the smallest natural number x that is c
  [I] # 2  msgs=20  target=                2396  Now I need to find $1008016 \mod 2431$.


  [C] # 3  msgs=20  target=                  60  Теперь у меня есть остаток 50760 от деления 102540 на 51780.
  [C] # 4  msgs= 5  target=              111915  The total number of chips produced over the two days is 111,
  [C] # 5  msgs= 5  target=              578336  The calculation shows that 578,336 barrels remain in the res
  [C] # 6  msgs= 5  target=            12598342  The total number of units shipped is 12,598,342.



12598342
  [C] # 7  msgs= 9  target=                1350  Multiplying 1351 by 6473 gives 8,745,023 residents, which ex
  [C] # 8  msgs= 5  target=                 354  The calculation shows that 1,000,000 mod 397 = 354.

This me
  [C] # 9  msgs= 7  target=             3984075  The hospital paid $3,984,075 afte

In [86]:
traces(log_toy_react)


Пример #0
Вопрос: ```Найдите наименьшее натуральное число x, которое сравнимо с 39 по модулю 51.```
[USER] ```Найдите наименьшее натуральное число x, которое сравнимо с 39 по модулю 51.```
[ASSISTANT] calls modular_arithmetic({'a': 39, 'b': 51})
Reasoning: ```Для решения задачи мне нужно найти наименьшее натуральное число x, такое что x ≡ 39 (mod 51).

Это означает, что x и 39 имеют одинаковый остаток при делении на 51. То есть x = 39 + 51k для некоторого целого k.

Поскольку мы ищем наименьшее натуральное число, я должен найти наименьшее положительное значение x.

Если k = 0, то x = 39, что уже является натуральным числом и меньше 51.

Проверим, является ли 39 наименьшим натуральным числом, удовлетворяющим условию:
x ≡ 39 (mod 51)

Это означает, что x = 39 + 51k, где k - целое число.

Для k = 0: x = 39
Для k = -1: x = 39 - 51 = -12 (не натуральное)
Для k = 1: x = 39 + 51 = 90 (больше 39)

Таким образом, наименьшее натуральное число x, которое сравнимо с 39 по модулю 51, равно 39.

Од

## Comparing the three approaches

Let's see the results side by side. Pay attention not just to accuracy but also to the
number of messages — more messages means more LLM calls, which means more cost and latency.

In [85]:
TOY_LOGS = [log_toy_gen, log_toy_naive, log_toy_react]
TOY_LABELS = ["generate only", "naive tool loop", "react v1"]

print(f"{'Approach':<25s} {'Acc':>5s}  {'Avg msgs':>8s}  {'Max msgs':>8s}")
print("-" * 50)
for label, log in zip(TOY_LABELS, TOY_LOGS):
    acc = get_acc(log)
    msg_list = [len(s.messages) for s in log.samples]
    avg_m = sum(msg_list) / len(msg_list)
    max_m = max(msg_list)
    print(f"{label:<25s} {acc:>4.0%}   {avg_m:>7.1f}   {max_m:>7d}")

Approach                    Acc  Avg msgs  Max msgs
--------------------------------------------------
generate only              47%       3.0         3
naive tool loop            87%       9.9        47
react v1                   87%       8.2        20


In [94]:
def print_trace(sample, title="Trace", max_content_len=200):
    """Выводит последовательность сообщений сэмпла в читаемом виде."""
    print(f"\n{'='*60}\n{title}\n{'='*60}")
    for i, msg in enumerate(sample.messages):
        role = msg.role.upper()
        if role == "SYSTEM":
            continue
        content_preview = (msg.content[:max_content_len] + "...") if len(msg.content) > max_content_len else msg.content
        print(f"[{i:2d}] {role:9s} | {content_preview}")
        
        # Проверяем, есть ли tool_calls и не пустой ли список
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"      -> TOOL CALL: {tc.function}({tc.arguments})")
        if role == "TOOL":
            print(f"      -> TOOL RESULT: {msg.content}")

In [95]:
sample_idx = 2  # 0-based индекс, Sample #3

sample_naive = log_toy_naive.samples[sample_idx]
sample_react = log_toy_react.samples[sample_idx]

print_trace(sample_naive, f"NAIVE LOOP - Sample #{sample_idx+1}")
print_trace(sample_react, f"REACT - Sample #{sample_idx+1}")


NAIVE LOOP - Sample #3
[ 1] USER      | Найди наибольший общий делить чисел 641060580 и 702705960
[ 2] ASSISTANT | To find the greatest common divisor (GCD) of the numbers 641060580 and 702705960, we can use the Euclidean algorithm. This algorithm involves repeatedly applying the operation of replacing the pair of...
      -> TOOL CALL: modular_arithmetic({'a': 641060580, 'b': 702705960})
[ 3] TOOL      | 641060580
      -> TOOL RESULT: 641060580
[ 4] ASSISTANT | The result of $641060580 \mod 702705960$ is 641060580, which means that 641060580 is less than 702705960. Now, we continue with the next step of the Euclidean algorithm by computing $702705960 \mod 64...
      -> TOOL CALL: modular_arithmetic({'a': 702705960, 'b': 641060580})
[ 5] TOOL      | 61645380
      -> TOOL RESULT: 61645380
[ 6] ASSISTANT | The result of $702705960 \mod 641060580$ is 61645380. We now continue with the Euclidean algorithm by computing $641060580 \mod 61645380$. Let's calculate this next step.


      -

---
1. Which approach performed best? Was it also the most expensive (most messages)?
2. Open the eval and compare the naive tool loop and the ReAct
   agent traces. Did the model actually use the tools in both cases, and how does the
   structure of the traces differ?

**Your answer:**

1. В моём случае наивный и ReAct подходы примерно одинаковые качество, но среднее кол-во вызов инструментов чуть больше у наивного подхода (видимо, за счёт выбросов --- есть случай с 47 вызовами)
2. В обоих случаях модели используют инструменты. Но в naive происходит одна длинная генерация. В react кждый вызов инструмента происходит в отдельном шаге цикла. После того как модель вызывает инструмент и получает результат, она заново оценивает ситуацию в новой генерации.

## 4. Adding a Symbolic Algebra Tool

Arithmetic tools help with computation, but many math problems require solving
equations — "find x such that 3x + 7 = 22". A small model can't do this reliably
without tools, but SymPy (a Python symbolic math library) can solve it exactly.

## Assignment 2: Create the `sympy_solve` tool

Implement a tool that takes an equation string (e.g. `"2*x + 5 = 21"` or `"3*x**2 - 12 = 0"`)
and returns the solutions using SymPy. The tool should:

1. Parse the equation — if it contains `=`, split into left and right sides and solve `left - right = 0`
2. If there's no `=`, treat the input as an expression equal to zero
3. Solve for the symbol `x`
4. Return the solutions as a string, or `"No solution found."` if empty
5. Handle errors gracefully

In [100]:
import sympy as sp
from inspect_ai.tool import tool

@tool
def sympy_solve():
    async def execute(equation: str) -> str:
        """Solve a symbolic equation or expression for the variable x.

        The input should be a string containing an equation like "2*x + 5 = 21",
        or an expression like "x**2 - 4" which is interpreted as equal to zero.
        The tool parses the equation, solves for x, and returns the set of solutions.

        Args:
            equation: A string representing the equation or expression to solve.
                      Examples: "3*x + 7 = 22", "x**2 - 9", "2*x + 5 = 3*x - 1".

        Returns:
            A string describing the solutions, e.g., "[5]" or "[-2, 2]".
            If no solution is found, returns "No solution found."
            If an error occurs during parsing or solving, returns an error message.
        """
        try:
            x = sp.Symbol('x')

            if '=' in equation:
                left_str, right_str = equation.split('=', 1)
                left_expr = sp.sympify(left_str.strip())
                right_expr = sp.sympify(right_str.strip())
                eq = sp.Eq(left_expr, right_expr)
            else:
                expr = sp.sympify(equation.strip())
                eq = sp.Eq(expr, 0)
            
            solutions = sp.solve(eq, x)
            if not solutions:
                return "No solution found."
            
            # Преобразуем решения в удобочитаемый вид
            sol_list = sorted(solutions, key=lambda s: sp.re(s) if s.is_real else sp.Abs(s))
            sol_str = "[" + ", ".join(str(s) for s in sol_list) + "]"
            return sol_str
            
        except sp.SympifyError as e:
            return f"Error: Could not parse equation. Invalid syntax. Details: {e}"
        except Exception as e:
            return f"Error: {type(e).__name__} - {e}"
    
    return execute

In [101]:
# Run the eval — the model should use your tool to solve both equations.
_log_sympy_test = eval(
    Task(
        dataset=[
            Sample(
                input="Solve for x: 2*x + 5 = 21. Reply with just the number.",
                target="8",
            ),
            Sample(
                input="Solve for x: x**2 - 9 = 0. What are the solutions? Reply with just the numbers separated by comma.",
                target="-3, 3",
            ),
        ],
        solver=react(
            prompt="You have a sympy_solve(equation) tool. Use it to solve the equation, then submit the result.",
            tools=[sympy_solve()],
            attempts=1,
        ),
        scorer=match(numeric=True),
        message_limit=10,
    ),
    model=MODEL,
)[0]

print_results("sympy_solve tool test", _log_sympy_test)

Output()

  sympy_solve tool test   accuracy: 100%
  [C] # 1  msgs= 5  target=                   8  I've solved the equation 2*x + 5 = 21 using the sympy_solve 
  [C] # 2  msgs= 5  target=               -3, 3  The solutions to the equation x² - 9 = 0 are x = -3 and x = 



In [102]:
traces(_log_sympy_test)


Пример #0
Вопрос: ```Solve for x: 2*x + 5 = 21. Reply with just the number.```
[USER] ```Solve for x: 2*x + 5 = 21. Reply with just the number.```
[ASSISTANT] calls sympy_solve({'equation': '2*x + 5 = 21'})
Reasoning: ```I need to solve the equation 2*x + 5 = 21 for x.

Let me use the sympy_solve tool to solve this equation.
```
[TOOL] returns: ```[8]```
[ASSISTANT] answers: ```I've solved the equation 2*x + 5 = 21 using the sympy_solve tool, and the solution is x = 8.

Let me verify this is correct by substituting back:
2*(8) + 5 = 16 + 5 = 21 ✓

The answer is correct.



8```

Пример #1
Вопрос: ```Solve for x: x**2 - 9 = 0. What are the solutions? Reply with just the numbers separated by comma.```
[USER] ```Solve for x: x**2 - 9 = 0. What are the solutions? Reply with just the numbers separated by comma.```
[ASSISTANT] calls sympy_solve({'equation': 'x**2 - 9 = 0'})
Reasoning: ```I need to solve the equation x² - 9 = 0 for x.

This is a quadratic equation that I can solve using the 

---
1. Did the model use the `sympy_solve` tool, or did it try to solve the equations
   in its head? (Check the message counts.)
2. If it didn't use the tool, try adjusting the prompt in the `react()` call.
   What wording helped?

**Your answer:**
1. Модель выбирает использовать инструмент  `sympy_solve` (логи ячейкой выше)
2. --

In [103]:
# Bundle all tools
ARITH_TOOLS = [add(), subtract(), multiply(), divide(), modular_arithmetic()]
ALL_TOOLS = ARITH_TOOLS + [sympy_solve()]

## 5. Loading the MATH-500 Benchmark

Now that we have a working agent with tools, let's evaluate it on a real benchmark.
**MATH-500** is a 500-question subset of the MATH dataset (Hendrycks et al., 2021),
covering competition-level math across seven subjects. It's available on Hugging Face.

Not all subjects benefit equally from our tools — Geometry and Counting & Probability
involve spatial reasoning and combinatorics that our calculator tools can't help with.
We'll focus on the four subjects where arithmetic and algebra tools are most relevant:
Algebra, Intermediate Algebra, Number Theory, and Prealgebra.

### Extracting answers from MATH solutions

MATH stores answers inside `\boxed{...}` in the solution string. We need a helper to
extract them:

In [104]:
TOOL_SUBJECTS = [
    "Algebra", "Number Theory", "Prealgebra", "Intermediate Algebra",
]


def extract_boxed(solution):
    """Extract the content of the last \\boxed{...} in a MATH solution string."""
    idx = solution.rfind("\\boxed{")
    if idx == -1:
        return solution.strip()
    start = idx + len("\\boxed{")
    depth = 1
    i = start
    while i < len(solution) and depth > 0:
        if solution[i] == "{":
            depth += 1
        elif solution[i] == "}":
            depth -= 1
        i += 1
    return solution[start:i - 1].strip()


def record_to_sample(record):
    """Convert a MATH-500 record into an inspect_ai Sample."""
    target = record.get("answer") or extract_boxed(record["solution"])
    return Sample(
        input=record["problem"],
        target=target,
        metadata={
            "level": int(record["level"]),
            "subject": record["subject"],
        },
    )

In [105]:
full_dataset = hf_dataset(
    path="HuggingFaceH4/MATH-500",
    split="test",
    sample_fields=record_to_sample,
    cached=True,
)

print(f"Total MATH-500: {len(full_dataset)} samples")

subject_counts = defaultdict(int)
for s in full_dataset:
    subject_counts[s.metadata["subject"]] += 1

print(f"\n{'Subject':<30s} {'Count':>5s}")
print("-" * 37)
for subj in sorted(subject_counts):
    marker = " <-- tool-friendly" if subj in TOOL_SUBJECTS else ""
    print(f"{subj:<30s} {subject_counts[subj]:>5d}{marker}")

Loading dataset HuggingFaceH4/MATH-500 from Hugging Face...


README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Total MATH-500: 500 samples

Subject                        Count
-------------------------------------
Algebra                          124 <-- tool-friendly
Counting & Probability            38
Geometry                          41
Intermediate Algebra              97 <-- tool-friendly
Number Theory                     62 <-- tool-friendly
Prealgebra                        82 <-- tool-friendly
Precalculus                       56


## Dev/test split

A core principle in evaluation: **never tune on your test set.** Iterating on the same
data you use for final scoring inflates results and makes them meaningless. We split the
tool-friendly subset into a small **dev set** (10%) for prompt and scaffolding iteration,
and a larger **test set** (90%) reserved for final evaluation only.

This mirrors the elicitation workflow described in [METR's Guidelines for Capability Elicitation](https://evaluations.metr.org/elicitation-protocol/): iterate against the dev set until failures stabilize, then run the test set once.

> **Adjust to your hardware.** If even the dev set takes too long, reduce `split_point`
> or set a smaller `EVAL_LIMIT`. The point is rapid iteration — you can always increase
> the test set size later.

In [106]:
tool_dataset = [s for s in full_dataset if s.metadata["subject"] in TOOL_SUBJECTS]
print(f"Tool-friendly subset: {len(tool_dataset)} samples")

random.seed(RANDOM_SEED)
random.shuffle(tool_dataset)

split_point = int(len(tool_dataset) * 0.1)
DEV_SET = tool_dataset[:split_point]
TEST_SET = tool_dataset[split_point:]

print(f"DEV_SET:  {len(DEV_SET)} samples")
print(f"TEST_SET: {len(TEST_SET)} samples")

Tool-friendly subset: 365 samples
DEV_SET:  36 samples
TEST_SET: 329 samples


## 6. Scoring Mathematical Answers

For the toy problems we used `match(numeric=True)`, which works when answers are plain
numbers. But MATH answers can be fractions (`3/7`), expressions (`2\sqrt{5}`), or
formatted in different equivalent ways — a simple string match will miss many correct answers.

We covered `model_graded_qa()` in notebook 3. Here we apply it to math: the key challenge
is writing a grading prompt that handles equivalent notations (e.g. `1/2` vs `0.5` vs `\frac{1}{2}`).
Note that in notebook 3 we deliberately hid the reference answer from the grading model — here
there's no reason to do that, so you can pass the correct answer directly.

> **Trade-off:** Model-graded scoring is slower and noisier, but catches equivalences that
> string matching misses. For production evals you might want a more capable model for
> grading than the one being tested — think about what makes sense for your setup.

## Assignment 3: Define the math scorer

Define a `math_scorer` using `model_graded_qa()`. Write a grading prompt that instructs
the model to judge mathematical equivalence and respond with **C** (correct) or **I** (incorrect),
and choose which model should do the grading. Think about what edge cases matter: different
notations, equivalent fractions, simplified vs unsimplified forms.

In [118]:
from inspect_ai.scorer import model_graded_qa
from inspect_ai.solver import TaskState

SCORER_MODEL = "openai/preview-code-pro"

GRADING_INSTRUCTIONS = """You are an expert mathematical grader. Compare the submitted answer with the correct answer and determine if they are mathematically equivalent.

Consider different notations:
- Fractions (1/2, 0.5, \\frac{1}{2})
- Radicals and exponents (\\sqrt{2} vs 2^{1/2})
- Factored vs expanded forms
- Decimal approximations (within reasonable rounding)

Respond with EXACTLY one letter: 'C' if equivalent, 'I' if not. Do not add any other text.
"""

MATH_SCORER = model_graded_qa(
    instructions=GRADING_INSTRUCTIONS,
    model=SCORER_MODEL,
    grade_pattern=re.compile(r"(C|I)"),
)

In [119]:
# Run the scorer on two toy samples where we know the answer.
_scorer_test_samples = [
    Sample(input="What is 1+1?", target="2"),
    Sample(input="What is 10/4?", target="5/2"),
]

_log_scorer_test = eval(
    Task(
        dataset=_scorer_test_samples,
        solver=[system_message("Answer the math question. Reply with just the answer."), generate()],
        scorer=MATH_SCORER,
    ),
    model=MODEL,
)[0]

print_results("Scorer sanity check", _log_scorer_test)
print("Check: does the scorer correctly mark equivalent answers as C?")

Output()

  Scorer sanity check   accuracy: 100%
  [C] # 1  msgs= 3  target=                   2  C
  [C] # 2  msgs= 3  target=                 5/2  C

Check: does the scorer correctly mark equivalent answers as C?


In [120]:
traces(_log_scorer_test)


Пример #0
Вопрос: ```What is 1+1?```
[USER] ```What is 1+1?```
[ASSISTANT] answers: ```2```

Пример #1
Вопрос: ```What is 10/4?```
[USER] ```What is 10/4?```
[ASSISTANT] answers: ```5/2 or 2.5```


---
1. Did your scorer correctly handle the fraction equivalence (`10/4` vs `5/2`)?
2. What failure modes can you imagine for model-graded scoring?

**Your answer:**
1. Да, обработано корректно.
2. Непредвиденный формат вывода, неоднозначность представления результатов (например, пустое множество).

## 7. Dev-Set Iteration — Building and Improving Your Agent

This is the core of agent evaluation: run on the dev set, see where the agent struggles,
and systematically improve it. The iteration loop:

1. Run the current agent on the dev set
2. Inspect failures — *why* did the agent get these wrong?
3. Hypothesize an improvement (better prompt? more tools? output formatting?) — and check
   whether the answer was actually correct but the scorer marked it wrong
4. Implement the change and re-evaluate on the dev set
5. Repeat

## Assignment 4: Build and improve a ReAct agent

Your goal is to build a ReAct agent and iterate on it using the dev set.
Does adding tools and scaffolding help on this task, and by how much?
Start by running a basic ReAct agent on the dev set, then look at the failures in the logs and iterate.

**Step 1 — Run a baseline.** Write a system prompt and run `react()` with `ALL_TOOLS` on the dev set.

**Step 2 — Inspect failures.** Open the logs and look at what went wrong. Common things to consider:
- Is the model ignoring tools and is failing?
- Is the answer mathematically correct but formatted wrong (e.g. `0.5` instead of `1/2`)?
- Is the model getting lost in multi-step problems?

**Step 3 — Iterate.** Based on what you see, try to improve. Some directions:
- Make the system prompt more explicit about strategy or answer format
- Add tools that cover operations the model struggles with (e.g. a single `calculator` tool, `gcd`, `factorial` or something very different)
- Add a format-extraction step after the `react()` loop if formatting is the main issue
- Reconsider the scorer if correct answers are being marked wrong

For each configuration you try, store the result as a `(description, log)` tuple in `DEV_RUNS` at the bottom — this will let you compare all your attempts in one table. Try at least two configurations.

In [123]:
MY_REACT_PROMPT = """
You are an expert mathematical problem solver with access to the following tools:
- add, subtract, multiply, divide: basic arithmetic operations.
- modular_arithmetic: compute a mod b (remainder).
- sympy_solve: solve symbolic equations or expressions for variable x.

Your task is to solve the given mathematical problem step by step using the available tools.
IMPORTANT GUIDELINES:
1. Break the problem into individual arithmetic or algebraic steps.
2. Use tools for EVERY calculation, no matter how simple it seems.
3. Do not perform mental math; always delegate to tools.
4. When you have the final answer, call the submit() tool with ONLY the answer.
5. The answer must be in the exact format expected by the problem (e.g., integer, fraction, expression).
6. If the problem involves solving an equation, use sympy_solve.
7. Think aloud before each tool call, but keep final answer concise.
"""

In [134]:
log_attempt_1 = eval(
    Task(
        dataset=DEV_SET,
        solver=react(
            prompt=MY_REACT_PROMPT,
            tools=ALL_TOOLS,
            attempts=1,
        ),
        scorer=MATH_SCORER,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
)[0]

print_results("Attempt 1 (baseline)", log_attempt_1)
DEV_RUNS = [("Attempt 1 (baseline)", log_attempt_1)]

Output()

  Attempt 1 (baseline)   accuracy: 90%
  [C] # 1  msgs= 3  target=                  10  C
  [C] # 2  msgs=11  target=                  47  C
  [C] # 3  msgs= 3  target=                2k+2  C
  [C] # 4  msgs= 3  target=                   2  C
  [C] # 5  msgs= 3  target=                   5  C
  [C] # 6  msgs= 3  target=          (a+5)(b+2)  C
  [C] # 7  msgs= 9  target=                 550  C
  [C] # 8  msgs= 5  target=                   2  C
  [C] # 9  msgs= 9  target=                   0  C
  [C] #10  msgs= 5  target=                   1  C
  [C] #11  msgs= 3  target=                   3  C
  [C] #12  msgs= 9  target=         \frac{3}{2}  C
  [C] #13  msgs= 3  target=                   1  C
  [C] #14  msgs= 5  target=                   4  C
  [C] #15  msgs= 5  target=                  14  C
  [C] #16  msgs= 9  target=               26000  C
  [C] #17  msgs=11  target=          \$32,\!348  C
  [C] #18  msgs= 5  target=                  13  C
  [C] #19  msgs= 3  target=                

In [135]:
# Look at the failures in the logs and decide what to change:
# prompt, tools, scorer, add a format-extraction step after react() or something else.
# Duplicate this cell for each new configuration. Give each log a new name.

In [136]:
MY_REACT_PROMPT_ADV = """
You are an expert mathematical problem solver with access to the following tools:
- add, subtract, multiply, divide: basic arithmetic operations.
- modular_arithmetic: compute a mod b (remainder).
- sympy_solve: solve symbolic equations or expressions for variable x.

Your task is to solve the given mathematical problem step by step using the available tools.
IMPORTANT GUIDELINES:
1. Break the problem into individual arithmetic or algebraic steps.
2. Use tools for EVERY calculation, no matter how simple it seems.
3. Do not perform mental math; always delegate to tools.
4. When you have the final answer, call the submit() tool with ONLY the answer.
5. The answer must be in the exact format expected by the problem (e.g., integer, fraction, expression).
6. If the problem involves solving an equation, use sympy_solve.
7. Think aloud before each tool call, but keep final answer concise.

CRITICAL RULES:
- You MUST use tools for EVERY arithmetic operation, even simple ones like 2+2. NEVER do mental math.
- After obtaining the final answer, you MUST call submit() with ONLY the answer, no extra text.
- For decimal period problems: find the order of 10 modulo the denominator using modular_arithmetic.
- For inequalities: do NOT use sympy_solve; instead solve the corresponding equation and test intervals.
- If a problem seems purely geometric and tools don't apply, describe a plan and reason step by step.

After obtaining the final answer, you MUST call submit() with ONLY the number or expression, and nothing else.
"""

In [137]:
log_attempt_2 = eval(
    Task(
        dataset=DEV_SET,
        solver=react(
            prompt=MY_REACT_PROMPT_ADV,
            tools=ALL_TOOLS,
            attempts=1,
        ),
        scorer=MATH_SCORER,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
)[0]

print_results("Attempt 2 (advanced)", log_attempt_2)
DEV_RUNS.append(("Attempt 2 (advanced)", log_attempt_2))

Output()

  Attempt 2 (advanced)   accuracy: 90%
  [C] # 1  msgs= 3  target=                  10  C
  [C] # 2  msgs=15  target=                  47  C
  [C] # 3  msgs= 7  target=                2k+2  C
  [C] # 4  msgs= 3  target=                   2  C
  [C] # 5  msgs= 5  target=                   5  C
  [C] # 6  msgs= 3  target=          (a+5)(b+2)  C
  [C] # 7  msgs= 9  target=                 550  C
  [C] # 8  msgs= 7  target=                   2  C
  [C] # 9  msgs= 5  target=                   0  C
  [C] #10  msgs= 3  target=                   1  C
  [C] #11  msgs= 3  target=                   3  C
  [C] #12  msgs= 5  target=         \frac{3}{2}  C
  [C] #13  msgs= 3  target=                   1  C
  [C] #14  msgs= 3  target=                   4  C
  [C] #15  msgs= 3  target=                  14  C
  [C] #16  msgs= 9  target=               26000  C
  [I] #17  msgs=20  target=          \$32,\!348  I
  [C] #18  msgs= 3  target=                  13  C
  [C] #19  msgs= 3  target=                

In [143]:
MY_REACT_PROMPT_STRICT = """
You are a mathematical agent. You MUST use tools for EVERY arithmetic or algebraic operation. Mental math is FORBIDDEN.

Available tools:
- add, subtract, multiply, divide
- modular_arithmetic(a, b): returns a mod b
- sympy_solve(equation): solves equation for x. Example: "2*x + 5 = 21"
- order_of_10(n): returns the order of 10 modulo n (use for repeating decimal problems)

CRITICAL RULES:
1. For ANY arithmetic (even 2+2) → call the appropriate tool.
2. For equations → use sympy_solve.
3. For repeating decimal period → use order_of_10(denominator). Do NOT manually compute powers.
4. After obtaining final answer → call submit(answer) with ONLY the number/expression.
5. For geometry problems without tools → reason step by step, but still use tools for any arithmetic.

Example WRONG: "I'll calculate 9*5=45" → CORRECT: call multiply(9,5)
Example WRONG: "10^5 = 100000 ≡ 1 (mod 11111)" → CORRECT: call order_of_10(11111)
"""

@tool
def order_of_10():
    async def execute(n: int) -> str:
        """Find the smallest positive integer k such that 10^k ≡ 1 (mod n).

        Args:
            n: The modulus (must be coprime to 10).

        Returns:
            The order of 10 modulo n as a string, or an error message.
        """
        import math
        if math.gcd(10, n) != 1:
            return "Error: n must be coprime to 10"
        k, val = 1, 10 % n
        while val != 1 and k <= n:
            val = (val * 10) % n
            k += 1
        return str(k) if val == 1 else f"Order > {n}"
    return execute

ALL_TOOLS_V2 = ALL_TOOLS + [order_of_10()]

In [150]:
log_attempt_3 = eval(
    Task(
        dataset=DEV_SET,
        solver=react(
            prompt=MY_REACT_PROMPT_STRICT,
            tools=ALL_TOOLS_V2,
            attempts=1,
        ),
        scorer=MATH_SCORER,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
)[0]

print_results("Attempt 3 (strict + tool)", log_attempt_3)
DEV_RUNS.append(("Attempt 3 (strict + tool)", log_attempt_3))

Output()

  Attempt 3 (strict + tool)   accuracy: 93%
  [C] # 1  msgs= 3  target=                  10  C
  [C] # 2  msgs=15  target=                  47  C
  [C] # 3  msgs= 3  target=                2k+2  C
  [C] # 4  msgs= 5  target=                   2  C
  [C] # 5  msgs= 5  target=                   5  C
  [C] # 6  msgs= 3  target=          (a+5)(b+2)  C
  [C] # 7  msgs= 9  target=                 550  C
  [C] # 8  msgs= 3  target=                   2  C
  [C] # 9  msgs= 7  target=                   0  C
  [C] #10  msgs= 5  target=                   1  C
  [C] #11  msgs= 3  target=                   3  C
  [C] #12  msgs= 7  target=         \frac{3}{2}  C
  [C] #13  msgs= 3  target=                   1  C
  [C] #14  msgs= 5  target=                   4  C
  [C] #15  msgs= 3  target=                  14  C
  [C] #16  msgs= 9  target=               26000  C
  [C] #17  msgs= 9  target=          \$32,\!348  C
  [C] #18  msgs= 3  target=                  13  C
  [C] #19  msgs= 3  target=           

In [151]:
print(f"{'Configuration':<40s} {'Dev Accuracy':>12s}")
print("=" * 54)
for description, log in DEV_RUNS:
    acc = get_acc(log)
    print(f"{description:<40s} {acc:>12.3%}")

Configuration                            Dev Accuracy
Attempt 1 (baseline)                          90.000%
Attempt 2 (advanced)                          90.000%
Attempt 3 (strict + tool)                     93.333%


In [152]:
def print_incorrect_samples(log, max_content_len=200):
    """Print details of samples that were scored as incorrect ('I')."""
    incorrect_samples = []
    for sample in log.samples:
        score = _first_score(sample)
        # score.value обычно 'C' или 'I' для model_graded_qa
        if score.value == 'I':
            incorrect_samples.append(sample)
    
    if not incorrect_samples:
        print("No incorrect samples found.")
        return
    
    print(f"Found {len(incorrect_samples)} incorrect samples out of {len(log.samples)} total.")
    print("=" * 80)
    
    for i, sample in enumerate(incorrect_samples, 1):
        score = _first_score(sample)
        original_idx = log.samples.index(sample)
        print(f"\n--- Incorrect Sample #{i} (original index: {original_idx}) ---")
        print(f"Question: {sample.input[:max_content_len]}...")
        print(f"Target: {sample.target}")
        
        # Попытаемся найти финальный ответ агента (последнее сообщение ассистента без tool_calls)
        model_answer = None
        for msg in reversed(sample.messages):
            if msg.role == "assistant" and not msg.tool_calls:
                model_answer = msg.content.strip()
                break
        if model_answer:
            print(f"Model answer (excerpt): {model_answer[:max_content_len]}...")
        else:
            print("Model answer: (not found)")
        
        print(f"Score explanation: {score.explanation}")
        print("-" * 60)

print_incorrect_samples(log_attempt_3)

Found 2 incorrect samples out of 30 total.

--- Incorrect Sample #1 (original index: 22) ---
Question: Let $z$ be a complex number such that $|z| = 1.$  Find the maximum value of
\[|1 + z| + |1 - z + z^2|.\]...
Target: \frac{13}{4}
Model answer (excerpt): I need to find the maximum value of $|1 + z| + |1 - z + z^2|$ where $|z| = 1$.

Since $|z| = 1$, I can write $z = e^{i\theta} = \cos\theta + i\sin\theta$ for some real $\theta$.

Let me first simplify...
Score explanation: I
------------------------------------------------------------

--- Incorrect Sample #2 (original index: 27) ---
Question: Let $T$ be the set of all triples $(a,b,c)$ of positive integers for which there exist triangles with side lengths $a,$ $b,$ $c.$  Compute
\[\sum_{(a,b,c) \in T} \frac{2^a}{3^b 5^c}.\]...
Target: \frac{17}{21}
Model answer (excerpt): Looking back at my work, I believe I've correctly computed the sum. Let me verify the key steps one more time:

From my derivation:
$$\frac{15}{56} \left(\frac{10}{

---
1. What modifications did you try? Which had the biggest impact?
2. What was the best dev-set accuracy you achieved? What configuration produced it?
3. Did any change that you expected to help actually hurt? Why might that be?
4. Look at the individual failures. What are the most common error types?

**Your answer:**
1. Изменил промпт (невызов инструментов, неправильное использование инструментов, слабое планирование, невызов submit, более строгий промпт) и добавил инструмент
2. 93% --- использовал всё, что описано в п.1
3. Вреда не было, но на втором шаге и не было прогресса. Видимо, улучшения не были достаточными
4. Вероятно не хватает знаний и навыков, которые можно было бы оформить инструментамию. Модель не находит инструмента и пытается решить самостоятельно

## 8. Test-Set Evaluation

You've iterated on the dev set and found your best configuration. Now it's time for the
moment of truth: evaluating on the held-out test set.

> **Run this section only once**, with your best configuration. Re-running and picking
> the best result would be "test-set contamination" — the same as peeking at a test set
> in ML.

## Assignment 5: Run and analyze your best configuration on the test set

Run your best agent configuration on the held-out test set, report accuracy with a 95%
confidence interval, and break down performance by subject and difficulty level. Note
anything that stands out.

In [153]:
def wilson_ci(n_correct, n_total, confidence_level=0.95):
    """Wilson score interval for a binomial proportion."""
    z = norm.ppf(0.5 + confidence_level / 2)
    p_hat = n_correct / n_total
    denom = 1 + z ** 2 / n_total
    centre = (p_hat + z ** 2 / (2 * n_total)) / denom
    margin = z * math.sqrt(
        (p_hat * (1 - p_hat) + z ** 2 / (4 * n_total)) / n_total
    ) / denom
    return max(0, centre - margin), min(1, centre + margin)

## Assignment 5.1: Run on the test set

Use your best configuration and run it to evaluate the test set

In [154]:
BEST_PROMPT = MY_REACT_PROMPT_STRICT

log_test = eval(
    Task(
        dataset=TEST_SET,
        solver=[
            react(
                prompt=BEST_PROMPT,
                tools=ALL_TOOLS_V2,
                attempts=1,
            ),
        ],
        scorer=MATH_SCORER,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=len(TEST_SET),
)[0]

Output()

In [155]:
# Подсчёт правильных ответов
n_test = len(log_test.samples)
n_correct_test = sum(1 for s in log_test.samples if _first_score(s).value == 'C')

lo, hi = wilson_ci(n_correct_test, n_test)
print(f"Test accuracy : {n_correct_test / n_test:.1%}")
print(f"95% Wilson CI : [{lo:.1%}, {hi:.1%}]")
print(f"n = {n_test}")

Test accuracy : 90.9%
95% Wilson CI : [87.3%, 93.5%]
n = 329


## Assignment 5.2: Breakdown by subject and difficulty level

There aren't enough samples per category to draw firm conclusions — that's fine.
Just see if anything stands out.

Iterate over logs and produce **two separate tables**: one grouped
by subject, one by difficulty level. 

Your output should look like this:

**By subject:**

| Subject | Correct | Total | Acc |
|---|---:|---:|---:|
| Algebra | 1 | 10 | 10% |
| Precalculus | 3 | 6 | 50% |
| ... | | | |

**By level:**

| Level | Correct | Total | Acc |
|---|---:|---:|---:|
| 1 | 4 | 5 | 80% |
| 2 | 3 | 6 | 50% |
| ... | | | |

The actual numbers will appear in the cell below.

In [156]:
subject_stats = defaultdict(lambda: [0, 0])
level_stats = defaultdict(lambda: [0, 0])

for sample in log_test.samples:
    sc = _first_score(sample)
    correct = 1 if sc.value == "C" else 0
    subj = sample.metadata.get("subject", "unknown")
    lvl = sample.metadata.get("level", 0)
    subject_stats[subj][0] += correct
    subject_stats[subj][1] += 1
    level_stats[lvl][0] += correct
    level_stats[lvl][1] += 1

print(f"{'Subject':<30s} {'Correct':>7s} {'Total':>5s} {'Acc':>6s}")
print("-" * 50)
for subj in sorted(subject_stats):
    c, t = subject_stats[subj]
    print(f"{subj:<30s} {c:>7d} {t:>5d} {c/t:>6.0%}")

print()
print(f"{'Level':<30s} {'Correct':>7s} {'Total':>5s} {'Acc':>6s}")
print("-" * 50)
for lvl in sorted(level_stats):
    c, t = level_stats[lvl]
    print(f"Level {lvl:<24d} {c:>7d} {t:>5d} {c/t:>6.0%}")

Subject                        Correct Total    Acc
--------------------------------------------------
Algebra                            112   113    99%
Intermediate Algebra                63    85    74%
Number Theory                       55    58    95%
Prealgebra                          69    73    95%

Level                          Correct Total    Acc
--------------------------------------------------
Level 1                             32    32   100%
Level 2                             57    59    97%
Level 3                             60    63    95%
Level 4                             79    85    93%
Level 5                             71    90    79%


**Your results:**

Fill in the tables once you've run the cell above.

**By subject:**

| Subject | Correct | Total | Acc |
|---|---:|---:|---:|
| Algebra | | | |
| Precalculus | | | |
| ... | | | |

**By difficulty level:**

| Level | Correct | Total | Acc |
|---|---:|---:|---:|
| 1 | | | |
| 2 | | | |
| ... | | | |

**By subject:**

| Subject | Correct | Total | Acc |
|---|---:|---:|---:|
| Algebra | 112 | 113 | 99% |
| Intermediate Algebra | 63 | 85 | 74% |
| Number Theory | 55 | 58 | 95% |
| Prealgebra | 69 | 73 | 95% |

**By difficulty level:**

| Level | Correct | Total | Acc |
|---|---:|---:|---:|
| 1 | 32 | 32 | 100% |
| 2 | 57 | 59 | 97% |
| 3 | 60 | 63 | 95% |
| 4 | 79 | 85 | 93% |
| 5 | 71 | 90 | 79% |

## Final comparison: dev vs test

In [157]:
print(f"{'Configuration':<30s} {'Dev Acc':>8s}  {'Test Acc':>8s}")
print("=" * 50)
for name, log in DEV_RUNS:
    acc = get_acc(log)
    print(f"{name:<30s} {acc:>8.0%}  {'--':>8s}")
print(f"{'best agent (TEST)':<30s} {'--':>8s}  {n_correct_test / n_test:>8.1%}")
print(f"\n95% CI on test: [{lo:.1%}, {hi:.1%}]")

Configuration                   Dev Acc  Test Acc
Attempt 1 (baseline)                90%        --
Attempt 2 (advanced)                90%        --
Attempt 3 (strict + tool)           93%        --
best agent (TEST)                    --     90.9%

95% CI on test: [87.3%, 93.5%]


---
1. How does the test accuracy compare to the dev accuracy? If there's a gap,
   what might explain it?
2. Which subjects and difficulty levels does the agent handle best? Worst?
3. Look at the confidence interval. Is it narrow enough to be useful, or would
   you want more test samples?
4. If you were to improve this agent further, where would you focus?

**Your answer:**

1. Различие есть, но незначительное, переобучения не произошло.
2. Лучше всего Algebra, хуже всего  Intermediate Algebra. Причина может быть в отсутствии подходящих инструментов. По уровням сложности --- чем сложнее, тем хуже результат
3. Доверительный интервал (87.3%, 93.5%) --- он достаточно широкий. В некоторых случаях нужен более узкий интервал, требуется больше примеров.
4. Добавить инструменты для Intermediate Algebra. Использовать более мощную модель

## Bonus assignment: Error Analysis

Pick 5-10 test samples that the agent got wrong and classify each failure. Here's one
possible taxonomy — feel free to use your own:

- **Tool misuse:** Called the wrong tool or with wrong arguments
- **Reasoning error:** Correct tool use but flawed multi-step reasoning
- **Format error:** Correct answer but wrong format in `submit()`
- **Inherent difficulty:** Problem requires reasoning beyond the tool set
- **Grading error:** The agent was actually right but the scorer got it wrong
- **Other:** Anything that doesn't fit the above

This kind of qualitative error analysis is essential in practice — aggregate metrics
tell you *how much* the agent struggles, but error analysis tells you *why*.

If you'd like to explore the logs in code, use the cell below — otherwise just read
them directly and delete it.

In [ ]:
# YOUR CODE HERE

---
1. What was the most common failure mode?
2. Which failure modes could be fixed with better prompting vs. better tools
   vs. a more capable model?
3. Did you find any grading errors? What does that imply about using model-graded scoring?

**Your answer:**